## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [209]:
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')


In [161]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [162]:
import numpy as np
import pandas as pd

In [163]:
data = pd.read_csv('prices.csv')

### Check all columns in the dataset

In [164]:
data.head()

,date,symbol,open,close,low,high,volume
0,2016-01-05 00:00:00,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06 00:00:00,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07 00:00:00,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08 00:00:00,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11 00:00:00,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


### Drop columns `date` and  `symbol`

In [165]:
data.drop(['date','symbol'],axis=1, inplace=True)

In [166]:
data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [167]:
data['volume'] = data['volume']/1000000

In [168]:
df = data.head(1000)

In [169]:
df

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086
5,115.510002,115.550003,114.500000,116.059998,1.0980
6,116.459999,112.849998,112.589996,117.070000,0.9496
7,113.510002,114.379997,110.050003,115.029999,0.7853
8,113.330002,112.529999,111.919998,114.879997,1.0937
9,113.660004,110.379997,109.870003,115.870003,1.5235


### Divide the data into train and test sets

In [185]:
from sklearn.model_selection import train_test_split
y = df['volume']
x = df.drop(['volume'], axis=1)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1)

#### Convert Training and Test Data to numpy float32 arrays


In [186]:
x_train = np.asarray(x_train)
y_train = np.asarray(y_train)

In [187]:
x_train.shape

(700, 4)

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [188]:
from sklearn import preprocessing

In [189]:
x_normalized = preprocessing.normalize(x, norm='l2')

In [190]:
x_normalized

array([[0.49582891, 0.50551007, 0.49132977, 0.50715709],
       [0.51032928, 0.4888958 , 0.4887328 , 0.51155174],
       [0.49941424, 0.49327777, 0.49319196, 0.5138328 ],
       ...,
       [0.49720098, 0.50510142, 0.49175846, 0.50580367],
       [0.49613772, 0.5051584 , 0.49331876, 0.50527118],
       [0.49263429, 0.50710743, 0.49167846, 0.5083363 ]])

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [191]:
#Input features
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')

#Normalize the data
x_n = tf.nn.l2_normalize(x,1) 

#Actual Prices
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')

W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

2.Define a function to calculate prediction

In [192]:
y = tf.add(tf.matmul(x_n,W),b,name='output') #X is normalized, hence X_n

3.Loss (Cost) Function [Mean square error]

In [193]:
loss = tf.reduce_mean(tf.square(y-y_),name='Loss')

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [194]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [195]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

### Get the shapes and values of W and b

In [196]:
print(W.shape)
print(b.shape)

(4, 1)
(1,)


### Model Prediction on 1st Examples in Test Dataset

In [197]:
for epoch in range(training_epochs):
            
    #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:x_train, y_:y_train}) # X and Y is like X_Train and Y_Train here.
    
    if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', train_loss)

Training loss at step:  0  is  225.87961
Training loss at step:  5  is  203.89143
Training loss at step:  10  is  197.76746
Training loss at step:  15  is  196.06197
Training loss at step:  20  is  195.58687
Training loss at step:  25  is  195.45464
Training loss at step:  30  is  195.41771
Training loss at step:  35  is  195.40744
Training loss at step:  40  is  195.40454
Training loss at step:  45  is  195.40382
Training loss at step:  50  is  195.40358
Training loss at step:  55  is  195.40346
Training loss at step:  60  is  195.40344
Training loss at step:  65  is  195.4035
Training loss at step:  70  is  195.40355
Training loss at step:  75  is  195.40353
Training loss at step:  80  is  195.40353
Training loss at step:  85  is  195.40349
Training loss at step:  90  is  195.40356
Training loss at step:  95  is  195.4036


In [184]:
test_loss = sess.run([loss],feed_dict={x:x_test, y_:y_test}) 
print ('Training loss at step:is ', test_loss)

Training loss at step:is  [57.884933]


## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [204]:
iris = pd.read_csv('11_Iris.csv')
iris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [205]:
ir = pd.get_dummies(iris,prefix=['Species'], columns=['Species'])

### Splitting the data into feature set and target set

In [206]:
from sklearn.model_selection import train_test_split

x =  ir.drop(['Id',"Species_Iris-setosa",'Species_Iris-versicolor',
       'Species_Iris-virginica'], axis=1)
y =  ir[['Species_Iris-setosa', 'Species_Iris-versicolor',
       'Species_Iris-virginica']]
xtrain,xtest,ytrain,ytest = train_test_split(x, y, test_size=0.30, random_state=1)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [210]:
from sklearn.neural_network import MLPClassifier
model=MLPClassifier(hidden_layer_sizes=(4))
model.fit(xtrain,ytrain)

MLPClassifier(activation='relu', alpha=0.0001, batch_size='auto', beta_1=0.9,
       beta_2=0.999, early_stopping=False, epsilon=1e-08,
       hidden_layer_sizes=4, learning_rate='constant',
       learning_rate_init=0.001, max_iter=200, momentum=0.9,
       nesterovs_momentum=True, power_t=0.5, random_state=None,
       shuffle=True, solver='adam', tol=0.0001, validation_fraction=0.1,
       verbose=False, warm_start=False)

### Model Training 

In [216]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential()
model.add(Dense(3, input_dim=4, kernel_initializer='normal', activation='softmax'))
#model.add(Dense(3, kernel_initializer='normal'))
# Compile model
model.compile(loss='categorical_crossentropy', optimizer='SGD', metrics= ['accuracy'])

### Model Prediction

In [217]:
model.fit(xtrain, ytrain, validation_data=(xtest,ytest), epochs=10, batch_size=5)
# evaluate the model
scores = model.evaluate(x, y)

Train on 105 samples, validate on 45 samples
Epoch 1/10
105/105 [==============================] - 0s 3ms/step - loss: 1.0273 - acc: 0.5048 - val_loss: 1.0198 - val_acc: 0.6000
Epoch 2/10
105/105 [==============================] - 0s 339us/step - loss: 0.9165 - acc: 0.6667 - val_loss: 0.9313 - val_acc: 0.6000
Epoch 3/10
105/105 [==============================] - 0s 453us/step - loss: 0.8383 - acc: 0.7048 - val_loss: 0.7965 - val_acc: 0.6000
Epoch 4/10
105/105 [==============================] - 0s 444us/step - loss: 0.7624 - acc: 0.7048 - val_loss: 0.7385 - val_acc: 0.7111
Epoch 5/10
105/105 [==============================] - 0s 359us/step - loss: 0.7176 - acc: 0.7238 - val_loss: 0.6945 - val_acc: 0.8667
Epoch 6/10
105/105 [==============================] - 0s 403us/step - loss: 0.6716 - acc: 0.7619 - val_loss: 0.7012 - val_acc: 0.6000
Epoch 7/10
105/105 [==============================] - 0s 310us/step - loss: 0.6457 - acc: 0.7048 - val_loss: 0.6415 - val_acc: 0.7111
Epoch 8/10
105/105 

In [218]:
scores[1]*100

66.66666666666666

### Save the Model

In [215]:
model.save("model.file")

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?